# Data Comparison
The objective of this stage is to perform a systematic reconciliation between the OMS and WMS datasets to identify all discrepancies and quantify their impact. Having standardized the data in the Validation phase, now we perform a direct "side-by-side" analysis to pinpoint exactly where the two systems diverge.

This step includes:
- Multi-level Matching: Using composite reconciliation key (`InvoiceNo` + `StockCode` + `Quantity`) to align records across both datasets at the most granular level.

- Completeness Gap Analysis: Full Outer Join to identify orphan records:
    - OMS-only records: transactions in OMS but missing in WMS (potential lost shipments)
    - WMS-only records: transactions in WMS but missing in OMS (potential ghost records)

- Numerical Variance Calculation: Quantifying differences in UnitPrice and other metrics for matched records.

- Duplicate & Error Impact: Analyzing how flagged records (duplicates, invalid dates, missing prices) affect reconciliation results.

- Financial Impact Assessment: Aggregating variances into total Net Variance showing financial exposure.

- Root Cause Cross-Referencing: Linking discrepancies back to Validation flags to identify systematic issues.


**Step 1** Data load

In [51]:
import pandas as pd
import ast

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
    
    # Konwertuj InvoiceDate na datetime
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
    
    # Konwertuj validation_flags ze stringa na listę
    df['validation_flags'] = df['validation_flags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
    
    return df

df_oms = load_data('validation/oms_final_clean.csv')
df_wms = load_data('validation/wms_final_clean.csv')

# Sprawdź flagi w WMS
wms_with_flags = df_wms[df_wms['validation_flags'].apply(lambda x: len(x) > 0)]
print(f"WMS records with flags: {len(wms_with_flags)}")
print(wms_with_flags[['InvoiceNo', 'StockCode', 'Quantity', 'validation_flags']].head(15))

# Sprawdź flagi w OMS
oms_with_flags = df_oms[df_oms['validation_flags'].apply(lambda x: len(x) > 0)]
print(f"\nOMS records with flags: {len(oms_with_flags)}")
print(oms_with_flags[['InvoiceNo', 'StockCode', 'Quantity', 'validation_flags']].head(15))


File validation/oms_final_clean.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File validation/wms_final_clean.csv loaded correctly using latin1 encoding. Loaded 542409 rows.
WMS records with flags: 12093
    InvoiceNo StockCode  Quantity     validation_flags
0      536365    85123A         6       [Invalid_Date]
1      536365     71053         6       [Invalid_Date]
2      536365    84406B         8       [Invalid_Date]
3      536365    84029G         6       [Invalid_Date]
4      536365    84029E         6       [Invalid_Date]
5      536365     22752         2       [Invalid_Date]
6      536365     21730         6       [Invalid_Date]
7      536366     22633         6       [Invalid_Date]
8      536366     22632         6       [Invalid_Date]
9      536367     84879        32       [Invalid_Date]
10     536367     22745         6       [Invalid_Date]
295    536396     21730         6  [Missing_UnitPrice]
485    536409     22111         1   [Duplicate_Record]
489    5

Observations:
- WMS contains 12,093 flagged records (2.23% of total), primarily consisting of duplicate records with some data quality issues (Invalid_Date, Missing_UnitPrice)
- OMS contains 10,147 flagged records (1.87% of total), exclusively duplicate records
- Invalid_Date flags in WMS (11 records) correspond to the intentional data corruption introduced at the beginning of the analysis
- The presence of flagged records in both datasets indicates that the Validation phase successfully identified data quality issues without removing them, allowing for impact analysis
- Duplicate records represent the largest category of flags in both systems, suggesting potential data entry errors or system synchronization issues

**Step 2** Multilevel matching  - data preparation for matching

In [56]:
# key for reconciliation
df_oms['reconciliation_key'] = df_oms['InvoiceNo'].astype(str) + '_' + df_oms['StockCode'].astype(str) + '_' + df_oms['Quantity'].astype(str)
df_wms['reconciliation_key'] = df_wms['InvoiceNo'].astype(str) + '_' + df_wms['StockCode'].astype(str) + '_' + df_wms['Quantity'].astype(str)

# Merge based on key
df_matched = pd.merge(df_oms, df_wms, on='reconciliation_key', how='inner', suffixes=('_oms', '_wms'))
print(f"Matched records: {len(df_matched)}")


Matched records: 555007


**Note**: 
A full outer join on the raw data showed an unnatural increase in the number of records (555,007 for databases with approximately 542,000). Analysis revealed the presence of many-to-many relationships (e.g., invoice 555524 generated 400 rows after the join). This is the result of mass duplication of identical order lines in the source systems.

In [59]:
# Checking the records that generated the most rows in the join
dupe_check = df_matched.groupby(['InvoiceNo_oms', 'StockCode_oms', 'Quantity_oms']).size().reset_index(name='row_count')
print(dupe_check.sort_values(by='row_count', ascending=False).head(10))

       InvoiceNo_oms StockCode_oms  Quantity_oms  row_count
207531        555524         22698             1        400
529092       C544580             S            -1        256
207530        555524         22697             1        144
408575        572861         22775            12         64
530714       C553531             S            -1         49
529094       C544583             S            -1         49
481701        578289         23395             1         36
25683         538514         21756             1         36
403039        572344             M            48         36
531661       C558347             S            -1         36


In [ ]:
print("\n AGGREGATING DUPLICATES")

# calculate duplicates before aggregation
oms_duplicates_count = len(df_oms) - df_oms[['InvoiceNo', 'StockCode', 'Quantity']].drop_duplicates().shape[0]
wms_duplicates_count = len(df_wms) - df_wms[['InvoiceNo', 'StockCode', 'Quantity']].drop_duplicates().shape[0]

print(f"OMS duplicates before aggregation: {oms_duplicates_count}")
print(f"WMS duplicates before aggregation: {wms_duplicates_count}")

# aggregation
df_oms_agg = df_oms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg({
    'UnitPrice': 'first',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': 'first'
}).reset_index()

df_wms_agg = df_wms.groupby(['InvoiceNo', 'StockCode', 'Quantity']).agg({
    'UnitPrice': 'first',
    'InvoiceDate': 'first',
    'CustomerID': 'first',
    'Country': 'first',
    'Description': 'first',
    'validation_status': 'first',
    'validation_flags': 'first'
}).reset_index()

print(f"\nOMS after aggregation: {len(df_oms_agg)} unique records")
print(f"WMS after aggregation: {len(df_wms_agg)} unique records")


 AGGREGATING DUPLICATES
OMS duplicates before aggregation: 5431
WMS duplicates before aggregation: 5931

OMS after aggregation: 536478 unique records
WMS after aggregation: 536478 unique records


In [61]:
df_matched = pd.merge(df_oms_agg, df_wms_agg, 
                      on=['InvoiceNo', 'StockCode', 'Quantity'],
                      how='inner', suffixes=('_oms', '_wms'))

print(f"Matched records: {len(df_matched)}")

Matched records: 536478
